In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

In [8]:
import torch
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

In [4]:
import scanpy as sc

In [6]:
import ray

In [7]:
import session_info
session_info.show()

/picb/lihonglab/chenxufeng/miniconda3/envs/scverse2/lib/python3.9/site-packages/session_info/main.py:213: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  mod_version = _find_version(mod.__version__)


In [31]:
adata = sc.read("/home/chenxufeng/picb_cxf/Data/scMultiSim/bench_grn/datasets/grn100_noise_tree1_1000_cells110_genes_sigma0.1_5/res.h5ad")

In [32]:
adata

AnnData object with n_obs × n_vars = 1000 × 110
    obs: 'cell_id', 'pop', 'depth', 'batch', 'n_counts', 'leiden', 'dpt_pseudotime', 'palantir_pseudotime'
    var: 'is_reg', 'is_target'
    uns: 'diffmap_evals', 'iroot', 'leiden', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_diffmap', 'X_pca', 'X_umap'
    varm: 'PCs', 'base_grn'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [33]:
adata.X.A

array([[0.50978434, 0.31905803, 0.31905803, ..., 0.        , 0.30849603,
        0.        ],
       [0.02974639, 0.4910665 , 0.37356058, ..., 0.        , 0.4142828 ,
        0.        ],
       [0.3896953 , 0.53929424, 0.13362074, ..., 0.        , 0.3055697 ,
        0.        ],
       ...,
       [0.18032414, 0.49427655, 0.10996456, ..., 0.04544688, 0.18998325,
        0.        ],
       [0.22527926, 0.34228623, 0.        , ..., 0.02873761, 0.17761338,
        0.        ],
       [0.14151858, 0.5302168 , 0.        , ..., 0.02006689, 0.18456559,
        0.        ]], dtype=float32)

In [29]:
adata.X.A.dtype

dtype('float64')

In [31]:
!ls /home/chenxufeng/picb_cxf/Data/NAFLD_HCC/1_AnnData/rna_sub/

B_cell.h5ad
Cholangiocyte.h5ad
Endothelial.h5ad
Endothelial_reanno_highres_fil.h5ad
Endothelial_reanno_highres.h5ad
Endothelial_scvi_withext1.h5ad
Epithelial.h5ad
Epithelial_scvi_withext1.h5ad
Hepatocyte_NRG1_scVI.h5ad
Hepatocyte_reanno_highres_fil.h5ad
Hepatocyte_reanno_highres.h5ad
Hepatocyte_reanno_highres_zonation.h5ad
Mesenchymal.h5ad
Mesenchymal_reanno_highres_fil.h5ad
Mesenchymal_reanno_highres.h5ad
Mesenchymal_scvi_withext1.h5ad
Myeloid.h5ad
Myeloid_reanno_highres_fil.h5ad
Myeloid_reanno_highres.h5ad
Myeloid_scvi_subcell.h5ad
Myeloid_scvi_withext1.h5ad
Myeloid_subcell_reanno_highres_fil.h5ad
Myeloid_subcell_reanno_highres.h5ad
Myeloid_subcell_scvi.h5ad
Myeloid_subnuclei_reanno_highres.h5ad
Neutrophil.h5ad
Proliferating.h5ad
T_cell.h5ad
T_cell_reanno_highres_fil.h5ad
T_cell_reanno_highres.h5ad
T_cell_scvi_withext1.h5ad


In [32]:
adata = sc.read("/home/chenxufeng/picb_cxf/Data/NAFLD_HCC/1_AnnData/rna_sub/Epithelial.h5ad")

In [33]:
adata.X.A.dtype

array([[0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        1.8432862],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       ...,
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ]], dtype=float32)

In [9]:
all_lags = torch.load("/home/chenxufeng/picb_cxf/Data/scMultiSim/bench_grn/datasets/grn100_noise_tree1_1000_cells110_genes_sigma0.1_5/tmp/all_lags.pt")

In [11]:
all_lags.shape

torch.Size([19, 110, 6, 5])

In [23]:
for i in range(len(all_lags)):
    for j in range(5):
        interactions = all_lags[i, :, :, j]
        nnz_percent = (all_lags[i,:,:,j] != 0).float().mean().data.numpy()
        print(f"Interaction {i} - {j}: {nnz_percent:.2%} non-zero values")

Interaction 0 - 0: 100.00% non-zero values
Interaction 0 - 1: 100.00% non-zero values
Interaction 0 - 2: 100.00% non-zero values
Interaction 0 - 3: 100.00% non-zero values
Interaction 0 - 4: 100.00% non-zero values
Interaction 1 - 0: 100.00% non-zero values
Interaction 1 - 1: 100.00% non-zero values
Interaction 1 - 2: 100.00% non-zero values
Interaction 1 - 3: 100.00% non-zero values
Interaction 1 - 4: 100.00% non-zero values
Interaction 2 - 0: 100.00% non-zero values
Interaction 2 - 1: 100.00% non-zero values
Interaction 2 - 2: 100.00% non-zero values
Interaction 2 - 3: 100.00% non-zero values
Interaction 2 - 4: 100.00% non-zero values
Interaction 3 - 0: 100.00% non-zero values
Interaction 3 - 1: 100.00% non-zero values
Interaction 3 - 2: 100.00% non-zero values
Interaction 3 - 3: 100.00% non-zero values
Interaction 3 - 4: 100.00% non-zero values
Interaction 4 - 0: 100.00% non-zero values
Interaction 4 - 1: 100.00% non-zero values
Interaction 4 - 2: 100.00% non-zero values
Interaction

In [29]:
from torch.nn.functional import normalize
def estimate_interactions(all_lags,lag=5,lower_thresh=0.01,upper_thresh=0.95,
						  binarize=False,l2_norm=False):

    all_interactions = []
    for i in range(len(all_lags)):
        for j in range(lag):

            nnz_percent = (all_lags[i,:,:,j] != 0).float().mean().data.numpy()

            if nnz_percent > lower_thresh and nnz_percent < upper_thresh:
                
                interactions = all_lags[i,:,:,j]

                if l2_norm:
                    interactions = normalize(interactions,dim=(0,1))
                if binarize:
                    interactions = (interactions != 0).float()
                
                all_interactions.append(interactions)
        
    return torch.stack(all_interactions).mean(0)

In [30]:
gc_mat = estimate_interactions(all_lags, lag=5, lower_thresh=0, upper_thresh=1.05,
						  binarize=False, l2_norm=False)

In [ ]:
gc_df = pd.DataFrame(gc_mat.cpu().data.numpy(),index=target_genes,columns=reg_genes)